# Hidraulična preša — predvidi, izračunaj, provjeri

**Poglavlje U01: fluid kao kontinuum, tlak i Pascalov zakon**

Osnovni scenarij ostaje idealna hidraulična preša. Uz račun sile pratit ćemo
pomak, rad i utjecaj mjernih nesigurnosti. Model pretpostavlja mirujući
nestlačivi fluid, krute klipove, zanemarivo trenje i jednak referentni tlak
s vanjske strane oba klipa.


## 1. Predvidi

Prije izvođenja koda zapiši svoje odgovore:

1. Ako se promjer izlaznog klipa udvostruči, koliko se puta mijenja sila $F_2$?
2. Ako mali klip prijeđe $s_1=80$ mm, je li pomak velikog klipa veći ili manji?
3. Koje će mjerenje više utjecati na $F_2$: pogreška od 1 % u promjeru ili 1 % u sili?

Tek nakon toga pokreni osnovni račun.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

def presa(D1_mm, D2_mm, F1_N, s1_mm=80.0):
    # Idealna preša; sve se interne veličine računaju u SI jedinicama.
    D1, D2 = np.asarray([D1_mm, D2_mm], dtype=float) / 1000.0
    A1, A2 = np.pi * D1**2 / 4.0, np.pi * D2**2 / 4.0
    p = F1_N / A1
    F2 = p * A2
    s2_mm = s1_mm * A1 / A2
    return {"A1": A1, "A2": A2, "p": p, "F2": F2,
            "s1_mm": s1_mm, "s2_mm": s2_mm}

osnovno = presa(D1_mm=32.0, D2_mm=160.0, F1_N=220.0, s1_mm=80.0)
for oznaka, vrijednost in osnovno.items():
    print(f"{oznaka:6s} = {vrijednost:.6g}")


## 2. Izračunaj — parametarska osjetljivost

Za idealni model vrijedi $F_2/F_1=(D_2/D_1)^2$, dok je
$s_2/s_1=(D_1/D_2)^2$. Mreža omjera promjera pokazuje kako se dobitak sile
plaća jednakim gubitkom pomaka. To je parametarska analiza, a ne nova formula.


In [ ]:
omjer_D = np.linspace(1.0, 7.0, 121)
dobitak_sile = omjer_D**2
omjer_pomaka = 1.0 / dobitak_sile

fig, ax1 = plt.subplots(figsize=(7.2, 4.2))
ax1.plot(omjer_D, dobitak_sile, color="#1565c0", label=r"$F_2/F_1$")
ax1.set(xlabel=r"omjer promjera $D_2/D_1$", ylabel="dobitak sile")
ax2 = ax1.twinx()
ax2.plot(omjer_D, omjer_pomaka, color="#c62828", label=r"$s_2/s_1$")
ax2.set_ylabel("omjer pomaka")
ax1.grid(ls=":", alpha=0.6)
fig.tight_layout()
plt.show()


## 3. Provjeri — bilance i mjerna nesigurnost

Prva neovisna provjera jest jednak tlak na oba klipa. Druga je jednakost
idealnih radova $F_1s_1=F_2s_2$. Za male, međusobno neovisne standardne
nesigurnosti linearizacija daje

$$
\left(\frac{u_{F_2}}{F_2}\right)^2=
\left(\frac{u_{F_1}}{F_1}\right)^2+
\left(2\frac{u_{D_2}}{D_2}\right)^2+
\left(2\frac{u_{D_1}}{D_1}\right)^2.
$$


In [ ]:
D1, D2, F1 = 32.0, 160.0, 220.0
u_D1, u_D2, u_F1 = 0.10, 0.20, 2.0  # mm, mm, N
r = presa(D1, D2, F1)
p_na_izlazu = r["F2"] / r["A2"]
W_ulaz = F1 * (r["s1_mm"] / 1000.0)
W_izlaz = r["F2"] * (r["s2_mm"] / 1000.0)

u_rel = np.sqrt((u_F1/F1)**2 + (2*u_D2/D2)**2 + (2*u_D1/D1)**2)
u_F2 = r["F2"] * u_rel
print(f"F2 = {r['F2']:.1f} ± {u_F2:.1f} N (standardna nesigurnost)")
print(f"idealni rad: ulaz {W_ulaz:.5f} J, izlaz {W_izlaz:.5f} J")

# Neovisne tvrdnje: prijenos tlaka, bilanca rada i granični slučaj.
assert np.isclose(r["p"], p_na_izlazu, rtol=1e-12)
assert np.isclose(W_ulaz, W_izlaz, rtol=1e-12)
assert np.isclose(presa(50, 50, 123)["F2"], 123.0, rtol=1e-12)
print("PASS: tlak, idealni rad i jednaki klipovi daju tri neovisne provjere.")


## Granica modela

U stvarnom uređaju trenje, deformacija crijeva, stlačivost fluida i razlika
visina smanjuju korisnu silu i mijenjaju pomak. Ovaj notebook provjerava
idealni Pascalov model; ne dokazuje nosivost ni sigurnost uređaja.
